In [ ]:
import numpy as np
import pandas as pd

def generate_wsvcr_data(n=500, censoring_rate=0.3, random_state=42):
    np.random.seed(random_state)

    X1 = np.random.normal(1, 1, n)
    X2 = np.random.normal(2, np.sqrt(2), n)
    X3 = np.random.exponential(1, n)

    X = np.column_stack([X1, X2, X3])

    X2_safe = np.where(np.abs(X2) < 1e-6, 1e-6, X2)

    lam = 5 * np.exp(
        -X1 * np.log(np.abs(X2_safe)) + np.sin(X3**2)
    )

    k = 3

    Z = np.random.weibull(k, n)
    Y = lam * Z

    c_max = np.quantile(Y, 1 - censoring_rate)
    C = np.random.uniform(0, c_max, n)

    T = np.minimum(Y, C)
    delta = (Y <= C).astype(int)

    l = np.zeros(n)
    u = np.zeros(n)

    for i in range(n):
        if delta[i] == 1:
            l[i] = T[i]
            u[i] = T[i]
        else:
            l[i] = T[i]
            u[i] = np.inf

    df = pd.DataFrame({
        "X1": X1,
        "X2": X2,
        "X3": X3,
        "Y_true": Y,
        "T_obs": T,
        "delta": delta,
        "l": l,
        "u": u
    })

    return X, l, u, df

In [11]:
import numpy as np
from cvxopt import matrix, solvers

solvers.options['show_progress'] = False

def wavelet_kernel(X, A=1.0):
    diff = X[:, None, :] - X[None, :, :]
    return np.prod(
        np.cos(1.75 * diff / A) * np.exp(-0.5 * (diff / A)**2),
        axis=2
    )

class WSVCR_QP:
    def __init__(self, C=1.0, A=1.0):
        self.C = C
        self.A = A

    def fit(self, X, l, u):
        n = X.shape[0]

        L = np.isfinite(l)
        U = np.isfinite(u)

        K = wavelet_kernel(X, self.A)

        P = np.block([
            [K, -K],
            [-K, K]
        ])

        q = np.zeros(2*n)

        for i in range(n):
            if L[i]:
                q[i] = -l[i]
            if U[i]:
                q[n + i] = u[i]

        A_eq = np.zeros((1, 2*n))

        for i in range(n):
            if L[i]:
                A_eq[0, i] = 1
            if U[i]:
                A_eq[0, n + i] = -1

        b_eq = np.array([0.0])

        G = np.vstack([
            np.eye(2*n),
            -np.eye(2*n)
        ])

        h = np.hstack([
            self.C * np.ones(2*n),
            np.zeros(2*n)
        ])

        P = matrix(P)
        q = matrix(q)
        G = matrix(G)
        h = matrix(h)
        A_eq = matrix(A_eq)
        b_eq = matrix(b_eq)

        solution = solvers.qp(P, q, G, h, A_eq, b_eq)

        z = np.array(solution['x']).flatten()

        self.alpha = z[:n]
        self.alpha_star = z[n:]
        self.X = X
        self.K = K

        self.b = self.compute_bias(l, u, L, U)

    def compute_bias(self, l, u, L, U):
        theta = self.alpha - self.alpha_star

        f = self.K @ theta

        b_vals = []

        for i in range(len(theta)):
            if L[i] and self.alpha[i] > 1e-5:
                b_vals.append(l[i] - f[i])
            elif U[i] and self.alpha_star[i] > 1e-5:
                b_vals.append(u[i] - f[i])

        return np.mean(b_vals) if b_vals else 0.0

    def predict(self, X_new):
        diff = self.X[:, None, :] - X_new[None, :, :]
        K_new = np.prod(
            np.cos(1.75 * diff / self.A) * np.exp(-0.5 * (diff / self.A)**2),
            axis=2
        )
        return (self.alpha - self.alpha_star) @ K_new + self.b

In [14]:
X, l, u, df = generate_wsvcr_data(
    n=500,
    censoring_rate=0.3,
    random_state=42
)

model = WSVCR_QP(C=2.0, A=1.0)
model.fit(X, l, u)

y_pred = model.predict(X)